# Flowers Knowledge RAG Pipeline
Step-by-step RAG demo using LangChain + Google Gemini + ChromaDB

## Step 1 - Install Dependencies

In [1]:
%pip install langchain langchain-community langchain-google-genai langchain-chroma langchain-text-splitters google-generativeai python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2 - Imports

In [2]:
import os
from dotenv import load_dotenv

# document loaders is used to load lot of documents in single run
# TextLoader because we are loading text files, for pdf use PDFLoader
from langchain_community.document_loaders import TextLoader

# to create chunks we use langchain_text_splitter
# Splitting text by recursively looking at characters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# we use chroma db to store the vectors, it will create vectors in our local laptop
from langchain_chroma import Chroma

# this we use to generate the embeddings and chat with Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

print("All libraries imported successfully")

/var/folders/d2/syj0jgl54sqgnjv2nbkk_wxc0000gn/T/ipykernel_14095/94182606.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


All libraries imported successfully


## Step 3 - Load API Key

In [3]:
load_dotenv(override=True)
api_key = os.environ.get("GOOGLE_API_KEY")
print("API Key loaded:", "OK" if api_key else "NOT FOUND - check your .env file")

API Key loaded: OK


## Step 4 - Load flowers.txt (Knowledge Base)

In [4]:
loader = TextLoader("flowers.txt", encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
print(f"Total characters: {len(documents[0].page_content)}")

Loaded 1 document(s)
Total characters: 36070


## Step 5 - Split Documents into Chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
docs = splitter.split_documents(documents)

print(f"Total chunks created: {len(docs)}")
print(f"\nSample chunk:\n{docs[0].page_content[:300]}")

Total chunks created: 55

Sample chunk:
Flowers, also known as blooms and blossoms, are the reproductive structures of flowering plants. Typically, they are structured in four circular levels around the end of a stalk. These include: sepals, which are modified leaves that support the flower; petals, often designed to attract pollinators; 


## Step 6 - Create Vector Database with Gemini Embeddings

In [6]:
vectorstore = Chroma.from_documents(
    docs,
    GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=api_key
    )
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector database created successfully")

Vector database created successfully


## Step 7 - Initialize Gemini LLM

In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.3,
    google_api_key=api_key
)

print("Gemini LLM initialized")

Gemini LLM initialized


## Step 8 - Build the RAG Function

In [8]:
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def ask_rag(question):
    # Step 1: Retrieve relevant chunks from vectorstore
    retrieved_docs = retriever.invoke(question)
    context = combine_docs(retrieved_docs)

    # Step 2: Build prompt with context + question
    prompt = f"""
You are a factual question-answering assistant.
Use ONLY the information provided in the context.
If the answer is not present, say: I don't know.

Context:
{context}

Question: {question}

Answer:
"""
    # Step 3: Get answer from Gemini LLM
    response = llm.invoke(prompt)
    content = response.content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content

print("RAG function ready")

RAG function ready


## Step 9 - Ask Questions

In [9]:
question = "What is the purpose of a flower?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")

Q: What is the purpose of a flower?
A: Based on the provided context, the main purpose of a flower is the reproduction of the individual, which aids in the survival of the species. Additionally, flowers produce spores (which become gametophytes that produce sex cells, leading to fertilized cells) and develop and help disseminate seeds.


In [10]:
question = "What are the parts of a flower?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")

Q: What are the parts of a flower?
A: Based on the provided context, the parts of a flower can be divided into vegetative (non-reproductive) parts and reproductive (sexual) parts, which are typically arranged in four whorls around a stalk:

* **Vegetative Parts (collectively known as the perianth):**
  * **Calyx (Sepals):** Modified, leaf-like outer structures that protect the developing flower.
  * **Petals:** Structures often designed to attract pollinators.

* **Reproductive (Sexual) Parts:**
  * **Male Parts (Androecium / Stamens):** Produce pollen. A stamen typically consists of an **anther** (which contains pollen sacs/thecae) connected to a **filament** (stalk).
  * **Female Parts (Gynoecium / Carpels or Pistil):** The innermost part of the flower. Each carpel consists of a **stigma** (which receives pollen), a **style** (the stalk), and an **ovary** (which contains the **ovules**).

* **Supporting Structures:**
  * **Receptacle:** The thickened tip of the flower stalk (**pedice

In [11]:
question = "What is pollination?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")

Q: What is pollination?
A: Based on the provided context, pollination is the movement of pollen from the male parts of a flower to the female parts.
